# Coordinates Module
The coordinates module provides structs and functions necessary for conversion between Geocentric (x, y z), Geodetic (Longitude, Latitude, Altitude), and Spherical coordinates and related measurements.

## External Data and License Notice
This guide will use geodetic coordinates from [OpenStreetMap](openstreetmap.org/copyright); please note that the data is available under the Open Database License. Please refer to the link for more information. The following code is not part of OpenStreeMap; it's provided by Trajectory under the MIT license. Please feel free to learn and contribute!

Our Coordinates module makes use of two amazing open source packages, please take a look at our source code for more information. Thanks! 

This notebook uses [GoNB](https://github.com/janpfeifer/gonb) for the Go integration. Please see the repo for instruction on how to set it up so you can run this notebook yourself.

## Example Uses

### Converting between Geodetic and Geocentric

Support for converstion between Geodetic (Longitude, Latitude, Altitude) and Geocentric (X, Y, Z) is provided by `GeodeticFromGeocentric()` and `GeocentricFromGeodetic()`.

Let's take a point on the [Sumida River](https://en.wikipedia.org/wiki/Sumida_River), one of my favorite spots in Tokyo. We'll put our coordinates in the `Geodetic` struct in the following order: Longitude, Latitude, Altitude.

In [68]:
// import packages
import (
	"fmt"
	"math"
	"github.com/go-gl/mathgl/mgl64"
    "github.com/trajectoryjp/geodesy_go/coordinates"
	"github.com/janpfeifer/gonb/cache"
	
)


var sumida = coordinates.Geodetic{
	139.78645,	// longitude
	35.67254,	// latitude
	0, 			// altitude  
}
%%
fmt.Print("%f", sumida)

%f[139.78645 35.67254 0]

We simply use `GeodeticFromGeocentric()` to convert to X, Y, Z format.

In [69]:

var sumidaGeocentric = coordinates.GeocentricFromGeodetic(sumida)

%%
fmt.Print("%.2f", sumidaGeocentric)

%.2f[-3.9612250964641897e+06 3.3491008647583686e+06 3.698736274363833e+06]

Converting back to Geodetic from Geocentric is just as simple.

In [70]:

var sumidaGeodetic = coordinates.GeodeticFromGeocentric(sumidaGeocentric)

%%
fmt.Printf("%.2f", sumidaGeodetic)

[139.79 35.67 0.00]

### Converting To Spherical and Measuring Distance
The `Spherical` type provides format for spherical systems. We have to convert to Spherical to measure the "as the bird flys" distance on the surface of the earth.

Let's add another point down the river, convert both to spherical, and measure the distance between the two.

Our module provides `GetLengthTo()`, which is a method on a spherical point that requires a second spherical point as an input.

In [71]:
 var tokyoBay = coordinates.Geodetic{
	139.76829,
	35.64328,
	0,
}

// convert both to Spherical
var sumidaSpherical = coordinates.Spherical(sumida)
var tokyoBaySpherical = coordinates.Spherical(tokyoBay)

// calculate the distance
var distance = sumidaSpherical.GetLengthTo(tokyoBaySpherical)

%%
fmt.Printf("sumidaSpherical: %f\nTokyoBaySpherical: %f\ndistance: %.2f meters", sumidaSpherical, tokyoBaySpherical, distance)

sumidaSpherical: [139.786450 35.672540 0.000000]
TokyoBaySpherical: [139.768290 35.643280 0.000000]
distance: 3647.92 meters

## Converting to Local from Geocentric

Let's convert `sumida` to a local point. Here's how we'll do the conversion:

1. Start with a point in `Geodetic{}` form.
2. Convert the geodetic point to geocentric.
3. Create a "geocentric -> local" conversion function using the `GenerateLocalFromGeocentric()` method of the geodetic point.
4. Use the conversion function from 3 to convert the geocentric point to local.

We already completed steps 1 and 2 above, so let's just use those values.


In [72]:
%%
fmt.Printf("sumida: %f\nsumidaGeocentric: %f", sumida, sumidaGeocentric)

sumida: [139.786450 35.672540 0.000000]
sumidaGeocentric: [-3961225.096464 3349100.864758 3698736.274364]

Next we convert use `sumida.GenerateLocalFromGeocentric()` method to create a function that will allow us to convert from Geocentric points to Local Points.

In [73]:
var ToLocalFunction = sumida.GenerateLocalFromGeocentric()

In step 4, we use this function to convert **geocentric** points to local. Notice, as with `tokyoBayLocal`, we can convert to geocentric directly inside the function call.

In [82]:
var sumidaLocal = ToLocalFunction(sumidaGeocentric)
var tokyoBayLocal = ToLocalFunction(coordinates.GeocentricFromGeodetic(tokyoBay))

%%
fmt.Printf("sumidaLocal: %f\ntokyoBayLocal: %f", sumidaLocal, tokyoBayLocal)

sumidaLocal: [-0.000000 -0.000000 0.000000]
tokyoBayLocal: [-1644.714662 -3246.309951 9.277925]

## Finding Distance between two local points

If we want to find the distance between two local points, we first create a difference vector using `mgl64.Vec3()` on the local point. This library has a `.Sub()` method that creates a resulting difference vector from another `mgl64.Vec3` point.


In [83]:
var vector1 = mgl64.Vec3(sumidaLocal)
var vector2 = mgl64.Vec3(tokyoBayLocal)
var differenceVector = vector1.Sub(vector2)

%%
fmt.Printf("%f-%f =%f", vector1, vector2, differenceVector)

[-0.000000 -0.000000 0.000000]-[-1644.714662 -3246.309951 9.277925] =[1644.714662 3246.309951 -9.277925]

Then we find the length of this vector:

In [84]:
var distance2 = differenceVector.Len()

%%
fmt.Printf("%f ", distance2)

3639.189565 

If we compare it from the results from `.GetLengthTo()` (from above in `distance`), the two distances are quite similar.

In [91]:
var difference = math.Abs(distance2 - distance)

%%
fmt.Printf("The difference between .GetLengthTo() and .Sub() methods is %f meters, or %.3f %%", difference, 100*difference/distance2)

The difference between .GetLengthTo() and .Sub() methods is 8.735089 meters, or 0.240 %